# Footlytics — 2D Tactical RadarTurn a full-pitch match video into tracked player positions on a scale pitch model.**Run order:** GPU check → install → weights → video → **calibrate** → run → inspect → render.The calibration step is the one that needs your attention. Everything after it is automatic,and everything after it is wrong if calibration is wrong.

## 1. Confirm the GPUAn A100 is plenty. Note the free memory — it decides whether the SoccerMaster jersey-number upgrade fits later (7B yes, 72B no).

In [ ]:
!nvidia-smiimport torch, platformprint(f"\ntorch {torch.__version__} | cuda {torch.version.cuda} | "      f"gpu {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")if torch.cuda.is_available():    print(f"vram {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

## 2. Install`ultralytics` brings the detector, `imageio-ffmpeg` writes the radar video.Everything else is already on Colab.

In [ ]:
%pip install -q ultralytics supervision imageio imageio-ffmpeg huggingface_hubimport cv2, numpy as np, pandas as pdprint("cv2", cv2.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

## 3. Get the Footlytics package onto ColabPick whichever applies. Option A once you have pushed this repo to GitHub;option B to work straight off Drive while iterating.

In [ ]:
# --- Option A: from GitHub (after you push) -------------------------------# !git clone https://github.com/<you>/Footlytics.git /content/Footlytics# --- Option B: from Google Drive ------------------------------------------from google.colab import drivedrive.mount('/content/drive')!cp -r "/content/drive/MyDrive/Footlytics" /content/Footlyticsimport syssys.path.insert(0, "/content/Footlytics")import footlyticsprint("footlytics loaded from", footlytics.__file__)

## 4. Detector weights`yolo_v8x6_finetuned.pt` ships with SoccerMaster on the Hugging Face Hub —one 195 MB file, no gate, no tracklab install. It is a YOLOv8x6 finetuned onSoccerNet, so it already knows what a footballer and a football look like.

In [ ]:
from huggingface_hub import hf_hub_downloadfrom footlytics.config import YOLO_REPO, YOLO_FILE, WEIGHTS_DIRweights = hf_hub_download(repo_id=YOLO_REPO, filename=YOLO_FILE,                          local_dir=str(WEIGHTS_DIR))print("weights at", weights)

### No footage yet? Fetch open dataAlfheim (Tromsø IL) is three *stationary* cameras covering the whole pitch, plus astitched panorama — the same optics as a Bepro rig — with ZXY body-sensor positionsat 20 Hz as ground truth. Fully open, no registration.Use `--view panorama` to exercise the tiled detector, or `--view 0` for a single1280x960 camera (about a tenth of the download).

In [ ]:
!apt-get -qq install -y ffmpeg > /dev/null!cd /content/Footlytics && python scripts/fetch_alfheim.py --minutes 2 --view panorama# The printed "video starts ..." timestamp is what aligns frames to the sensor# ground truth later — copy it.VIDEO = "/content/Footlytics/data/raw/alfheim/alfheim_2013-11-03_panorama.mp4"VIDEO_START = "2013-11-03 18:01:12.794293"   # <- from the fetch outputZXY = "/content/Footlytics/data/raw/alfheim/zxy_2013-11-03_panorama.parquet"

## 5. Point at a match videoBest case is a fixed full-pitch view (Bepro-style). A broadcast clip also works,but the camera moves, so a single fixed calibration is **not** valid for it —see the note in section 6.

In [ ]:
VIDEO = "/content/drive/MyDrive/Footlytics/data/match.mp4"   # <-- edit meimport cv2, osassert os.path.exists(VIDEO), f"not found: {VIDEO}"cap = cv2.VideoCapture(VIDEO)W  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));  H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))N  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT));  FPS = cap.get(cv2.CAP_PROP_FPS)print(f"{W}x{H}  {N} frames  {FPS:.2f} fps  =  {N/FPS/60:.1f} min")CALIB_FRAME = min(500, max(N // 2, 0))       # a frame with clear pitch markingscap.set(cv2.CAP_PROP_POS_FRAMES, CALIB_FRAME)ok, bgr = cap.read(); cap.release()assert ok, "could not read the calibration frame"frame = bgr[:, :, ::-1].copy()import matplotlib.pyplot as pltplt.figure(figsize=(18, 10)); plt.imshow(frame); plt.axis("off")plt.title(f"frame {CALIB_FRAME} — the frame you will calibrate on"); plt.show()

## 6. Calibrate — the step that mattersClick each prompted landmark. Scroll to zoom (a corner flag is a few pixels wide),drag to pan.**Aim for 10–15 landmarks spread across the whole frame.** Measured on a syntheticcamera: 4 clustered points → ~1 m error; 20 spread points → ~0.2 m. Points bunchedin one half give a homography that explodes at the far end.**If your footage is broadcast (a camera that pans/zooms), stop here.** One fixedhomography is only valid for a fixed camera. You would need per-frame calibration —that is what SoccerMaster's `KeypointsDetection` head is for, and it is thePhase-1 upgrade.

In [ ]:
from footlytics.geometry import annotateannotate.click_landmarks(frame)

Paste the printed `IMAGE_POINTS` below (or, in Colab, just run — it is picked up automatically from the click tool).

In [ ]:
from footlytics.geometry import annotatefrom footlytics.geometry.pitch import PitchIMAGE_POINTS = annotate.LAST_CLICKS or {    # fallback: paste the block printed under the canvas, e.g.    # "corner_LT": (312.0, 486.5),}assert len(IMAGE_POINTS) >= 4, "click at least 4 landmarks (aim for 10+)"# Measure your actual pitch if you can; V.League grounds are not all 105x68.PITCH = Pitch(length=105.0, width=68.0)# use_tps=True only for a STITCHED panorama, where seams bend straight lines and# one homography leaves systematic residuals. On synthetic stitched footage it# cut mean error from 1.25 m to 0.22 m; on a true single camera it just fits noise.cal = annotate.build_calibration(IMAGE_POINTS, frame, pitch=PITCH, use_tps=False)cal.save("/content/Footlytics/calib/venue.json")

### Check it visuallyNumbers can look fine while the mapping is subtly wrong. The yellow model linesmust sit **on** the painted lines, at both ends. If they drift at the far end,add landmarks there and refit.

In [ ]:
annotate.overlay_pitch(frame, cal, pitch=PITCH)import matplotlib.pyplot as plt; plt.show()

## 7. Run the pipelineStart with a short clip (`max_frames`) to confirm quality before committing to90 minutes.`stride=2` halves the work and is usually fine for tactical analysis: playersdo not change shape meaningfully in 40 ms. Drop to `stride=1` for ball trackingor anything about acceleration.

In [ ]:
from footlytics.perception.detect import Detector, DetectorConfigfrom footlytics.pipeline.radar import run, PipelineConfigfrom footlytics.state.schema import MatchMeta, TeamInfodet = Detector(weights, DetectorConfig(    tile=1280 if max(W, H) > 2200 else 0,   # tile only when the frame is big    overlap=0.25, conf=0.25, ball_conf=0.10, device="cuda", half=True,))meta = MatchMeta(    match_id="VN-DEMO-001", fps=FPS,    pitch_length=PITCH.length, pitch_width=PITCH.width,    home=TeamInfo("Đội nhà", "HOME"), away=TeamInfo("Đội khách", "AWAY"),    source_type="fixed_panoramic",           # drives the sanity checks    competition="V.League demo",)state, report = run(    VIDEO, cal, det, meta,    cfg=PipelineConfig(stride=2, max_frames=750, use_appearance=True),)

## 8. Inspect before believing`validate()` is deliberately blunt. A tracking bug looks exactly like a tacticalinsight until someone checks the units.

In [ ]:
print(state)print(f"\nprocessed {report['fps_processed']:.2f} frames/s")print(f"detections: {report['detections']:,}  "      f"(dropped off-pitch: {report['dropped_off_pitch']:,})")print(f"tracks created: {report['tracks_created']}")print(f"median players per frame: {report['median_players_per_frame']:.1f}  (expect ~22)")print("\nvalidation:")probs = report["validation"]print("  clean" if not probs else "\n".join(f"  - {p}" for p in probs))if "team_diagnosis" in report:    print("\nteams:", report["team_diagnosis"]["summary"])

### Confirm the team split by eye`diagnose()` screens out matches that are definitely broken; it does **not**certify the rest. Synthetic white-vs-white (64% accurate) and red-vs-orange(100% accurate) scored almost identically. Ten seconds of looking settles it.

In [ ]:
import matplotlib.pyplot as pltclf = report.get("_team_clf")if clf is not None:    frames_, boxes_, tids_, descs_ = report["_montage"]    m = clf.cluster_montage(frames_, boxes_, tids_, descs_, per_team=10)    fig, axes = plt.subplots(2, 10, figsize=(16, 5))    for r, (team, crops) in enumerate(m.items()):        for c in range(10):            axes[r, c].axis("off")            if c < len(crops): axes[r, c].imshow(crops[c])        axes[r, 0].set_title(team, loc="left", fontsize=11)    plt.suptitle("each row should be one team — if they are mixed, team stats are unsafe")    plt.tight_layout(); plt.show()

### One frame of the radar

In [ ]:
from footlytics.viz.radar import plot_frameimport matplotlib.pyplot as pltfig, ax = plt.subplots(figsize=(14, 9), dpi=110)fig.patch.set_facecolor("#0d1117")plot_frame(state, frame_idx=state.n_frames // 2, ax=ax, trail_frames=25)plt.show()

## 9. Render the radar clipThis is the artefact you put in front of a coach.

In [ ]:
from footlytics.viz.radar import render_videoout = render_video(state, "/content/radar.mp4", start=0, end=min(state.n_frames, 500),                   trail_frames=15)from IPython.display import HTMLfrom base64 import b64encodeHTML(f'<video width=900 controls src="data:video/mp4;base64,'     f'{b64encode(open(out,"rb").read()).decode()}">')

## 10. Physical numbersSprint thresholds here follow the senior convention (HSR > 5.5 m/s, sprint > 7.0 m/s).Lower them for youth football — they are conventions, not physics.Distances scale with clip length, so these are only comparable across playerswithin the same clip.

In [ ]:
from footlytics.analytics.kinematics import distance_summarysummary = distance_summary(state.players, state.meta.fps)display(summary.head(25))

### Save the Match StateEverything downstream — formations, pressing, xT, reports — reads this and nevertouches a GPU again.

In [ ]:
path = state.save("/content/drive/MyDrive/Footlytics/data/out/VN-DEMO-001")print("saved to", path)from footlytics.state.schema import MatchStateprint(MatchState.load(path))

---## What to fix first, in order1. **`median players per frame` well below 22** → detection, not tracking. Lower   `conf`, or reduce `tile` to 960 so distant players are bigger relative to the crop.2. **Warnings about positions off the pitch** → calibration. Re-run section 6 with   more landmarks near the far touchline.3. **Impossible speeds** → identity switches. Confirm `use_appearance=True`; if kits   clash, jersey numbers are the real fix (SoccerMaster's Qwen2.5-VL-7B module).4. **Mixed team montage** → kit clash. Assign teams by hand for the demo and note it.Detector quality is the binding constraint. In synthetic testing, once detectionsdegraded past ~0.6 m of error, no tracker improvement recovered the result.

---## Score against ground truth (Alfheim only)Everything else in this notebook tells you the code ran. This tells you whether itwas *right*, in metres, against body-sensor positions.Only Tromsø players wore sensors, so unmatched detections are mostly theopposition, not false positives — recall is meaningful here, precision is not.

In [ ]:
import sys; sys.path.insert(0, "/content/Footlytics/scripts")from score_vs_zxy import score, verdictfrom datetime import datetimeimport pandas as pds = score(state.players, pd.read_parquet(ZXY),          datetime.fromisoformat(VIDEO_START), state.meta.fps)for k, v in s.items(): print(f"{k:18s} {v}")print("\n" + verdict(s))